# Inverse Protocol Prediction (IPP) — All 5 Models

This notebook trains and evaluates all five IPP architectures from the SLiMIA-IPP paper:

| Model | Backbone | Key Design |
|-------|----------|------------|
| ConvNeXt-Tiny | ImageNet pretrained | Convolutional locality priors |
| ViT-B/16 | Pretrained timm weights | Global self-attention via CLS token |
| CoAtNet-0 | Trained from scratch | Hybrid convolution + attention |
| ImageShapeFusion | ConvNeXt-Tiny + shape tokens | Explicit morphometric priors |
| HMTT | ViT-B/16 encoder | Causal label conditioning |

**Data split:** Technical replicate level (T1–T4 train, T5+T8 val, T6+T7+T9–T24 test)  
**Evaluation:** 3 independent seeds → mean ± std  

> The ImageShapeFusion model requires a shape features CSV (see `03_morphometry_extraction.ipynb`).

In [ ]:
# !pip install timm --quiet

In [ ]:
# Imports
import os
import random
import math
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import tifffile
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
import joblib

## Configuration

In [ ]:
# Config
class CFG:
    # Paths
    metadata_csv = "../data/slimia_metadata.csv"
    shape_csv    = "../data/shape_features_with_metadata.csv"  # needed for fusion model
    ckpt_dir     = "../checkpoints/ipp/"
    output_dir   = "../results/ipp/"

    # Label columns — 8 protocol attributes
    # We include all 9 here for completeness; per-label results reported separately.
    label_columns = [
        "microscope", "cell_line", "culture_medium", "formation_method",
        "seeding_density", "timepoint", "biological_rep", "magnification"
    ]

    # Shape features (from automated RefineNet segmentation)
    shape_features = [
        "area", "perimeter", "eccentricity", "solidity", "extent",
        "equivalent_diameter", "major_axis_length", "minor_axis_length", "circularity"
    ]

    # Train/val/test technical replicate assignments (Table in paper)
    train_reps = ["T1", "T2", "T3", "T4"]
    val_reps   = ["T5", "T8"]
    test_reps  = ["T6", "T7"] + [f"T{i}" for i in range(9, 25)]

    # Fusion transformer hyperparams
    d_model   = 256
    n_heads   = 4
    n_layers  = 3
    ff_dim    = 512
    dropout   = 0.1

    # Training
    image_size  = 224
    batch_size  = 32
    num_epochs  = 200
    patience    = 40
    lr          = 1e-4
    weight_decay= 1e-2
    num_workers = 2
    seeds       = [42, 123, 999]
    device      = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(CFG.ckpt_dir,  exist_ok=True)
os.makedirs(CFG.output_dir, exist_ok=True)
print(f"Device: {CFG.device}")

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## Data Loading & Splits

Splits are done at the **technical replicate** level — no image from a given acquisition session
appears in more than one split. This prevents acquisition-specific information leakage.

In [ ]:
# Load metadata and encode labels 
df = pd.read_csv(CFG.metadata_csv)
df["full_path"] = df["full_path"].astype(str).str.strip()

# Label encoding — fit on full dataset so all classes are mapped consistently
label_encoders = {}
for col in CFG.label_columns:
    le = LabelEncoder()
    df[col] = df[col].astype(str)
    df[col + "_enc"] = le.fit_transform(df[col])
    label_encoders[col] = le
    joblib.dump(le, os.path.join(CFG.output_dir, f"label_encoder_{col}.pkl"))

# Technical replicate splits
train_df = df[df["technical_rep"].isin(CFG.train_reps)].copy()
val_df   = df[df["technical_rep"].isin(CFG.val_reps)].copy()
test_df  = df[df["technical_rep"].isin(CFG.test_reps)].copy()

# Sanity checks — no replicate appears in two splits
assert set(train_df["technical_rep"]).isdisjoint(set(val_df["technical_rep"]))
assert set(train_df["technical_rep"]).isdisjoint(set(test_df["technical_rep"]))
assert set(val_df["technical_rep"]).isdisjoint(set(test_df["technical_rep"]))

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
label_dims = {col: df[col + "_enc"].nunique() for col in CFG.label_columns}
print("Label dims:", label_dims)

In [ ]:
# Image loading helper 
def load_image_rgb(path: str) -> Image.Image:
    """Loads .ome.tiff or standard image formats as RGB PIL image."""
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        arr = tifffile.imread(path)
        if arr.ndim == 2:
            arr = np.stack([arr] * 3, axis=-1)
        elif arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)
        return Image.fromarray(arr.astype(np.uint8)).convert("RGB")

In [ ]:
# Standard IPP Dataset (image to labels)
class IPPDataset(Dataset):
    """
    Used by ConvNeXt, ViT, CoAtNet, and HMTT.
    Returns (image_tensor, label_tensor) where label_tensor is shape [L].
    """
    def __init__(self, frame: pd.DataFrame, label_cols: list, transform=None):
        self.df        = frame.reset_index(drop=True)
        self.label_cols = label_cols
        self.transform  = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = load_image_rgb(row["full_path"])
        if self.transform:
            img = self.transform(img)
        labels = torch.tensor([int(row[c + "_enc"]) for c in self.label_cols],
                               dtype=torch.long)
        return img, labels


# Fusion Dataset (image + shape vector → labels)
class FusionIPPDataset(Dataset):
    """
    Used by ImageShapeFusionTransformer.
    Returns (image_tensor, shape_vector, label_tensor).
    """
    def __init__(self, frame: pd.DataFrame, label_cols: list,
                 shape_cols: list, transform=None):
        self.df        = frame.reset_index(drop=True)
        self.label_cols = label_cols
        self.shape_cols = shape_cols
        self.transform  = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row.get("full_path", row.get("image_path", ""))
        img  = load_image_rgb(path)
        if self.transform:
            img = self.transform(img)
        shape_vec = torch.tensor(
            row[self.shape_cols].to_numpy(dtype=np.float32))
        labels = torch.tensor([int(row[c + "_enc"]) for c in self.label_cols],
                               dtype=torch.long)
        return img, shape_vec, labels


# Transforms 
# IPP uses conservative augmentation (paper section: morphology-preserving)
train_tf = T.Compose([
    T.Resize((CFG.image_size, CFG.image_size)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3),
])
val_tf = T.Compose([
    T.Resize((CFG.image_size, CFG.image_size)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3),
])

## Model Architectures

In [ ]:
# Multi-task backbone models (ConvNeXt, ViT, CoAtNet) 
class MultiTaskBackbone(nn.Module):
    """
    Generic multi-head wrapper for any timm backbone.
    One linear head per protocol label.
    Used for ConvNeXt-Tiny, ViT-B/16, and CoAtNet-0.
    """
    def __init__(self, backbone_name: str, label_dims: dict, pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0)
        in_features = self.backbone.num_features
        self.heads   = nn.ModuleDict({
            label: nn.Linear(in_features, dim)
            for label, dim in label_dims.items()
        })

    def forward(self, x):
        feats = self.backbone(x)
        return {label: head(feats) for label, head in self.heads.items()}

In [ ]:
# ImageShapeFusionTransformer
class ImageShapeFusionTransformer(nn.Module):
    """
    ConvNeXt-Tiny backbone + 9 per-feature shape tokens fused via a
    Transformer encoder. d_model=256, 3 layers, 4 heads.

    Each of the 9 morphometric features (area, perimeter, etc.) becomes
    a separate token, giving the model explicit access to geometric cues.
    Shape features are z-normalised using training set statistics.
    """
    def __init__(self, label_dims: dict, shape_mean: np.ndarray,
                 shape_std: np.ndarray):
        super().__init__()
        n_shape = len(CFG.shape_features)
        D       = CFG.d_model

        # Image branch
        self.backbone   = timm.create_model("convnext_tiny", pretrained=True,
                                            num_classes=0)
        self.image_proj = nn.Linear(self.backbone.num_features, D)

        # Shape tokens (one Linear per feature, scalar → D)
        self.shape_proj = nn.ModuleList([nn.Linear(1, D) for _ in range(n_shape)])
        self.n_shape    = n_shape

        # Positional embeddings: 1 image + n_shape tokens
        self.pos_embed  = nn.Parameter(torch.randn(1, 1 + n_shape, D) * 0.02)

        # Transformer encoder
        enc_layer       = nn.TransformerEncoderLayer(
            d_model=D, nhead=CFG.n_heads, dim_feedforward=CFG.ff_dim,
            dropout=CFG.dropout, activation="gelu")
        self.transformer = nn.TransformerEncoder(enc_layer,
                                                  num_layers=CFG.n_layers)
        self.norm        = nn.LayerNorm(D)

        # Classification heads
        self.heads = nn.ModuleDict({
            label: nn.Sequential(
                nn.LayerNorm(D), nn.Linear(D, D), nn.GELU(),
                nn.Dropout(0.2), nn.Linear(D, ncls)
            ) for label, ncls in label_dims.items()
        })

        # Store normalisation stats as buffers (saved with the model)
        self.register_buffer("shape_mean",
                             torch.tensor(shape_mean, dtype=torch.float32))
        self.register_buffer("shape_std",
                             torch.tensor(shape_std,  dtype=torch.float32))

    def forward(self, images, shape_feats):
        # Image → token
        img_tok = self.image_proj(self.backbone(images)).unsqueeze(1)  # (B,1,D)

        # Shape → tokens
        shape_n = (shape_feats - self.shape_mean) / (self.shape_std + 1e-6)
        s_toks  = torch.cat([self.shape_proj[i](shape_n[:, i:i+1]).unsqueeze(1)
                             for i in range(self.n_shape)], dim=1)  # (B,n,D)

        seq  = torch.cat([img_tok, s_toks], dim=1) + self.pos_embed  # (B,1+n,D)
        fused = self.norm(self.transformer(seq.permute(1,0,2))[0])    # (B,D)
        return {label: head(fused) for label, head in self.heads.items()}

In [ ]:
# HMTT: Hierarchical Multi-Task Transformer
class HMTT(nn.Module):
    """
    ViT-B/16 encoder + hierarchical label conditioning.

    At training time: teacher forcing — each head receives the ground-truth
    embeddings of all preceding labels as context.
    At eval time:    autoregressive — greedy predictions from earlier heads
                     are used as context for later heads.

    Label order (causal, matches paper):
      cell_line → culture_medium → seeding_density → magnification →
      microscope → timepoint → biological_rep → (technical_rep if present)
    """
    def __init__(self, label_dims: dict, label_order: list):
        super().__init__()
        self.label_order = label_order
        self.all_labels  = list(label_dims.keys())

        self.encoder  = timm.create_model("vit_base_patch16_224",
                                          pretrained=True, num_classes=0)
        D = self.encoder.num_features

        # Per-label embeddings for context conditioning
        self.embeds = nn.ModuleDict({
            lab: nn.Embedding(label_dims[lab], D)
            for lab in self.all_labels
        })

        # Classification heads: [feat ‖ context] → logits
        self.heads = nn.ModuleDict({
            lab: nn.Sequential(
                nn.LayerNorm(D * 2),
                nn.Linear(D * 2, D), nn.GELU(), nn.Dropout(0.2),
                nn.Linear(D, label_dims[lab])
            ) for lab in self.all_labels
        })

    def forward(self, x, labels=None, teacher_forcing: bool = True):
        B     = x.size(0)
        feat  = self.encoder(x)   # (B, D)
        zeros = torch.zeros(B, feat.size(1), device=x.device)

        outputs      = {}
        chosen       = {}   # greedy predictions for autoregressive eval

        for i, lab in enumerate(self.label_order):
            # Build context from all preceding labels in the causal order
            if i == 0:
                ctx = zeros
            else:
                ctx = zeros.clone()
                for j in range(i):
                    prev = self.label_order[j]
                    if teacher_forcing and labels is not None:
                        idx = labels[:, self.all_labels.index(prev)]
                    else:
                        idx = chosen[prev]
                    ctx = ctx + self.embeds[prev](idx)

            logits        = self.heads[lab](torch.cat([feat, ctx], dim=1))
            outputs[lab]  = logits
            chosen[lab]   = logits.argmax(1)

        # Any labels not in the causal order: predict flat (no context)
        for lab in self.all_labels:
            if lab not in self.label_order:
                outputs[lab] = self.heads[lab](torch.cat([feat, zeros], dim=1))

        return outputs

## Focal Loss & Class Weights

In [ ]:
# Losses 
class FocalLoss(nn.Module):
    """Focal loss (Lin et al. 2017). Used by HMTT alongside class-weighted CE."""
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha  # per-class weight tensor, or None

    def forward(self, logits, target):
        ce   = F.cross_entropy(logits, target, reduction="none")
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        if self.alpha is not None:
            a    = self.alpha[target] if self.alpha.ndim == 1 else self.alpha
            loss = a * loss
        return loss.mean()


def build_class_weights(train_frame: pd.DataFrame,
                        label_dims: dict) -> dict:
    """Compute balanced class weights per label from the training split."""
    weights = {}
    n = len(train_frame)
    for col in CFG.label_columns:
        y  = train_frame[col + "_enc"].values
        nc = label_dims[col]
        counts = np.bincount(y, minlength=nc).astype(np.float32)
        w = np.zeros(nc, dtype=np.float32)
        for i in range(nc):
            if counts[i] > 0:
                w[i] = n / (nc * counts[i])
        weights[col] = torch.tensor(w, device=CFG.device)
    return weights

## Training & Evaluation Helpers

In [ ]:
# Metrics helper
def compute_metrics(targets: dict, preds: dict) -> dict:
    """Compute per-label and macro-averaged acc/prec/rec/f1."""
    per_label = {}
    for col in CFG.label_columns:
        t, p = np.array(targets[col]), np.array(preds[col])
        per_label[col] = dict(
            acc  = accuracy_score(t, p),
            prec = precision_score(t, p, average="macro", zero_division=0),
            rec  = recall_score(t, p, average="macro", zero_division=0),
            f1   = f1_score(t, p, average="macro", zero_division=0),
        )
    macro = {k: np.mean([per_label[c][k] for c in CFG.label_columns])
             for k in ["acc","prec","rec","f1"]}
    return {"per_label": per_label, "macro": macro}


# Generic train-one-epoch
def _train_epoch(model, loader, criterions, optimizer, scaler,
                 is_hmtt=False, is_fusion=False):
    model.train()
    t_preds, t_tgts = defaultdict(list), defaultdict(list)

    for batch in tqdm(loader, desc="  train", leave=False):
        if is_fusion:
            imgs, shape, labels = batch
            imgs   = imgs.to(CFG.device)
            shape  = shape.to(CFG.device)
            labels = labels.to(CFG.device)
        else:
            imgs, labels = batch
            imgs   = imgs.to(CFG.device)
            labels = labels.to(CFG.device)
            shape  = None

        optimizer.zero_grad()
        with torch.amp.autocast(device_type=CFG.device):
            if is_hmtt:
                outs = model(imgs, labels=labels, teacher_forcing=True)
            elif is_fusion:
                outs = model(imgs, shape)
            else:
                outs = model(imgs)

            loss = sum(criterions[c](outs[c], labels[:, i])
                       for i, c in enumerate(CFG.label_columns))

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        for i, c in enumerate(CFG.label_columns):
            t_preds[c] += outs[c].argmax(1).detach().cpu().tolist()
            t_tgts[c]  += labels[:, i].detach().cpu().tolist()

    return compute_metrics(t_tgts, t_preds)


@torch.no_grad()
def _eval_epoch(model, loader, criterions,
                is_hmtt=False, is_fusion=False):
    model.eval()
    v_preds, v_tgts = defaultdict(list), defaultdict(list)

    for batch in tqdm(loader, desc="  val", leave=False):
        if is_fusion:
            imgs, shape, labels = batch
            imgs  = imgs.to(CFG.device)
            shape = shape.to(CFG.device)
            labels = labels.to(CFG.device)
        else:
            imgs, labels = batch
            imgs   = imgs.to(CFG.device)
            labels = labels.to(CFG.device)
            shape  = None

        if is_hmtt:
            outs = model(imgs, labels=None, teacher_forcing=False)
        elif is_fusion:
            outs = model(imgs, shape)
        else:
            outs = model(imgs)

        for i, c in enumerate(CFG.label_columns):
            v_preds[c] += outs[c].argmax(1).cpu().tolist()
            v_tgts[c]  += labels[:, i].cpu().tolist()

    return compute_metrics(v_tgts, v_preds)

In [ ]:
# Generic training loop
def train_model(model_name: str, model: nn.Module, train_loader, val_loader,
                criterions: dict, save_path: str,
                is_hmtt=False, is_fusion=False,
                use_adamw=False) -> dict:
    """
    Trains a model to completion with early stopping.
    Returns training history (per-epoch macro F1 for train and val).
    """
    model = model.to(CFG.device)

    if use_adamw:
        optimizer = torch.optim.AdamW(model.parameters(),
                                      lr=CFG.lr, weight_decay=CFG.weight_decay)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=CFG.lr)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=5, factor=0.5)
    scaler    = torch.amp.GradScaler(enabled=(CFG.device == "cuda"))

    best_f1     = 0.0
    p_counter   = 0
    history     = dict(train_f1=[], val_f1=[], val_acc=[])

    for epoch in range(1, CFG.num_epochs + 1):
        tr_m = _train_epoch(model, train_loader, criterions, optimizer, scaler,
                            is_hmtt, is_fusion)
        va_m = _eval_epoch(model, val_loader, criterions, is_hmtt, is_fusion)

        tr_f1 = tr_m["macro"]["f1"]
        va_f1 = va_m["macro"]["f1"]
        va_acc = va_m["macro"]["acc"]

        history["train_f1"].append(tr_f1)
        history["val_f1"].append(va_f1)
        history["val_acc"].append(va_acc)

        scheduler.step(va_f1)
        print(f"  Epoch {epoch:03d} | Train F1 {tr_f1:.4f} | "
              f"Val F1 {va_f1:.4f} | Val Acc {va_acc:.4f}")

        if va_f1 > best_f1:
            best_f1   = va_f1
            p_counter = 0
            torch.save(model.state_dict(), save_path)
            print(f"  ✓ Saved best (F1 {best_f1:.4f})")
        else:
            p_counter += 1
            if p_counter >= CFG.patience:
                print(f"  Early stopping at epoch {epoch}.")
                break

    return history

## Run All Models

In [ ]:
# Standard loaders (ConvNeXt, ViT, CoAtNet, HMTT)
train_loader = DataLoader(
    IPPDataset(train_df, CFG.label_columns, train_tf),
    batch_size=CFG.batch_size, shuffle=True,
    num_workers=CFG.num_workers, pin_memory=True)

val_loader = DataLoader(
    IPPDataset(val_df, CFG.label_columns, val_tf),
    batch_size=CFG.batch_size, shuffle=False,
    num_workers=CFG.num_workers, pin_memory=True)

test_loader = DataLoader(
    IPPDataset(test_df, CFG.label_columns, val_tf),
    batch_size=CFG.batch_size, shuffle=False,
    num_workers=CFG.num_workers, pin_memory=True)

In [ ]:
# HMTT label order (causal, from paper)
HMTT_ORDER = [
    "cell_line", "culture_medium", "seeding_density", "magnification",
    "microscope", "timepoint", "biological_rep"
]
# Keep only labels that are in label_columns
HMTT_ORDER = [l for l in HMTT_ORDER if l in CFG.label_columns]

class_weights = build_class_weights(train_df, label_dims)

In [ ]:
# Model registry 
# (name, factory_fn, criterions, use_adamw, is_hmtt, is_fusion)
def make_ce_criterions():
    return {c: nn.CrossEntropyLoss() for c in CFG.label_columns}

def make_weighted_ce_criterions():
    return {c: nn.CrossEntropyLoss(weight=class_weights[c])
            for c in CFG.label_columns}

MODEL_REGISTRY = [
    (
        "ConvNeXt-Tiny",
        lambda: MultiTaskBackbone("convnext_tiny", label_dims, pretrained=True),
        make_ce_criterions,
        False, False, False,  # adamw, hmtt, fusion
    ),
    (
        "ViT-B16",
        lambda: MultiTaskBackbone("vit_base_patch16_224", label_dims, pretrained=True),
        make_ce_criterions,
        False, False, False,
    ),
    (
        "CoAtNet-0",
        lambda: MultiTaskBackbone("coatnet_0_224", label_dims, pretrained=False),
        make_ce_criterions,
        False, False, False,
    ),
    # Fusion and HMTT are trained separately below (need shape CSV / label order)
]

In [ ]:
# Training loop (3 seeds, backbone models) 
all_results = {}

for model_name, model_fn, crit_fn, use_adamw, is_hmtt, is_fusion in MODEL_REGISTRY:
    print(f"\n{'='*60}\n  {model_name}\n{'='*60}")
    seed_results = []

    for seed in CFG.seeds:
        print(f"\n  --- Seed {seed} ---")
        set_seed(seed)

        model     = model_fn()
        criterions = crit_fn()
        save_path = os.path.join(CFG.ckpt_dir,
                                 f"{model_name}_seed{seed}.pth")

        history = train_model(
            model_name, model, train_loader, val_loader,
            criterions, save_path,
            is_hmtt=is_hmtt, is_fusion=is_fusion,
            use_adamw=use_adamw)

        # Evaluate best checkpoint on test set
        model.load_state_dict(torch.load(save_path, map_location=CFG.device))
        test_m = _eval_epoch(model, test_loader, criterions, is_hmtt, is_fusion)
        seed_results.append(test_m["macro"])
        print(f"  Test macro: {test_m['macro']}")

    all_results[model_name] = seed_results

In [ ]:
# ImageShapeFusionTransformer 
print("\n" + "="*60 + "\n  ImageShapeFusionTransformer\n" + "="*60)

# Merge metadata + shape features
meta_df_  = pd.read_csv(CFG.metadata_csv)
shape_df_ = pd.read_csv(CFG.shape_csv)
meta_df_["fname"]  = meta_df_["full_path"].apply(lambda x: Path(x).name)
shape_df_["fname"] = shape_df_["image_path"].apply(lambda x: Path(x).name)
merged = pd.merge(meta_df_, shape_df_, on="fname", how="inner")

# Fix _x/_y columns from merge
for col in CFG.label_columns:
    if col + "_x" in merged.columns:
        merged[col] = merged[col + "_x"]
merged = merged.drop(columns=[c for c in merged.columns if c.endswith("_x") or c.endswith("_y")])

for col in CFG.label_columns:
    merged[col] = merged[col].astype(str)
    merged[col + "_enc"] = label_encoders[col].transform(merged[col])

for sf in CFG.shape_features:
    merged[sf] = pd.to_numeric(merged[sf], errors="coerce").fillna(0.0)

fus_train = merged[merged["technical_rep"].isin(CFG.train_reps)].copy()
fus_val   = merged[merged["technical_rep"].isin(CFG.val_reps)].copy()
fus_test  = merged[merged["technical_rep"].isin(CFG.test_reps)].copy()

shape_mean = fus_train[CFG.shape_features].mean().values.astype(np.float32)
shape_std  = fus_train[CFG.shape_features].std().replace(0,1).values.astype(np.float32)
np.save(os.path.join(CFG.output_dir, "shape_mean.npy"), shape_mean)
np.save(os.path.join(CFG.output_dir, "shape_std.npy"),  shape_std)

fus_train_loader = DataLoader(
    FusionIPPDataset(fus_train, CFG.label_columns, CFG.shape_features, train_tf),
    batch_size=CFG.batch_size, shuffle=True,
    num_workers=CFG.num_workers, pin_memory=True)
fus_val_loader = DataLoader(
    FusionIPPDataset(fus_val,   CFG.label_columns, CFG.shape_features, val_tf),
    batch_size=CFG.batch_size, shuffle=False,
    num_workers=CFG.num_workers, pin_memory=True)
fus_test_loader = DataLoader(
    FusionIPPDataset(fus_test,  CFG.label_columns, CFG.shape_features, val_tf),
    batch_size=CFG.batch_size, shuffle=False,
    num_workers=CFG.num_workers, pin_memory=True)

fus_class_wts = build_class_weights(fus_train, label_dims)

fus_seed_results = []
for seed in CFG.seeds:
    print(f"\n  --- Seed {seed} ---")
    set_seed(seed)
    model     = ImageShapeFusionTransformer(label_dims, shape_mean, shape_std)
    criterions = {c: nn.CrossEntropyLoss(weight=fus_class_wts[c])
                  for c in CFG.label_columns}
    save_path = os.path.join(CFG.ckpt_dir, f"FusionTransformer_seed{seed}.pth")
    train_model("FusionTransformer", model,
                fus_train_loader, fus_val_loader,
                criterions, save_path,
                is_fusion=True, use_adamw=True)
    model.load_state_dict(torch.load(save_path, map_location=CFG.device))
    test_m = _eval_epoch(model, fus_test_loader, criterions, is_fusion=True)
    fus_seed_results.append(test_m["macro"])

all_results["ImageShapeFusion"] = fus_seed_results

In [ ]:
# HMTT 
print("\n" + "="*60 + "\n  HMTT\n" + "="*60)

hmtt_seed_results = []
for seed in CFG.seeds:
    print(f"\n  --- Seed {seed} ---")
    set_seed(seed)
    model     = HMTT(label_dims, HMTT_ORDER)
    criterions = {c: nn.CrossEntropyLoss(weight=class_weights[c])
                  for c in CFG.label_columns}
    save_path = os.path.join(CFG.ckpt_dir, f"HMTT_seed{seed}.pth")
    train_model("HMTT", model, train_loader, val_loader,
                criterions, save_path,
                is_hmtt=True, use_adamw=True)
    model.load_state_dict(torch.load(save_path, map_location=CFG.device))
    test_m = _eval_epoch(model, test_loader, criterions, is_hmtt=True)
    hmtt_seed_results.append(test_m["macro"])

all_results["HMTT"] = hmtt_seed_results

## Results Summary (Table 5)

In [ ]:
# Aggregate: mean ± std across 3 seeds
rows = []
for model_name, seed_metrics in all_results.items():
    row = {"Model": model_name}
    for metric in ["acc", "prec", "rec", "f1"]:
        vals = [m[metric] for m in seed_metrics]
        row[metric] = f"{np.mean(vals):.4f} ± {np.std(vals):.5f}"
    rows.append(row)

results_df = pd.DataFrame(rows).set_index("Model")
print("\nIPP Results (mean ± std, seeds 42 / 123 / 999)")
print(results_df.to_string())

results_df.to_csv(os.path.join(CFG.output_dir, "ipp_results.csv"))
print("\nSaved to", os.path.join(CFG.output_dir, "ipp_results.csv"))